#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [1]:
#@markdown <br><center><img src='https://static.vecteezy.com/system/resources/previews/012/871/368/non_2x/google-drive-icon-google-product-illustration-free-png.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive


#`Setup And Update ComfyUI`



In [2]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

!mkdir -p /content/drive/MyDrive/AI_Models_for_Video/diffusion_models /content/drive/MyDrive/AI_Models_for_Video/text_encoders /content/drive/MyDrive/AI_Models_for_Video/vae
!rm -rf /content/ComfyUI/models
!ln -s /content/drive/MyDrive/AI_Models_for_Video /content/ComfyUI/models

-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 65458, done.
remote: Counting objects: 100% (1067/1067), done.
remote: Compressing objects: 100% (387/387), done.
remote: Total 65458 (delta 904), reused 685 (delta 680), pack-reused 64391 (from 2)
Receiving objects: 100% (65458/65458), 100.09 MiB | 17.12 MiB/s, done.
Resolving deltas: 100% (45368/45368), done.
/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu118, https://download.pytorch.org/whl/cu117
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 217.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 202.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 MB 180.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 141.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100

# `Models Download`

## Download MODELS

In [3]:
# @title Загрузка моделей Wan 2.2
STORAGE = "Временная сессия Colab" # @param ["Временная сессия Colab", "Google Drive"]
DRIVE_FOLDER = "AI_Models_for_Video" # @param {type:"string"}

import os, shutil
from google.colab import userdata

%cd {WORKSPACE}
!apt-get -y install -qq aria2

# --- Токен Civitai (нужен только для Civitai) ---
try:
    CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
except Exception:
    CIVITAI_API_TOKEN = None
print("Civitai token:", "✅" if CIVITAI_API_TOKEN else "❌ нет (не нужен для HuggingFace)")

# --- Выбор хранилища ---
MODELS_LINK = f"{WORKSPACE}/models"
SUBDIRS = ["checkpoints", "diffusion_models", "text_encoders", "vae",
           "loras", "clip_vision", "controlnet", "upscale_models"]

if STORAGE == "Google Drive":
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount('/content/drive')
    MODELS_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    os.makedirs(MODELS_DIR, exist_ok=True)
    for d in SUBDIRS:
        os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)

    # заменяем models в ComfyUI симлинком на Диск
    if os.path.islink(MODELS_LINK):
        os.unlink(MODELS_LINK)
    elif os.path.isdir(MODELS_LINK):
        shutil.rmtree(MODELS_LINK)
    os.symlink(MODELS_DIR, MODELS_LINK)
    print(f"📁 Модели будут сохранены на Google Диск: {MODELS_DIR}")
else:
    MODELS_DIR = MODELS_LINK
    # если раньше был симлинк на Диск, убираем его и делаем обычную папку
    if os.path.islink(MODELS_LINK):
        os.unlink(MODELS_LINK)
    for d in SUBDIRS:
        os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)
    print(f"⚡ Модели будут сохранены во временную сессию: {MODELS_DIR}")
    print("   После закрытия сессии Colab они пропадут.")

# --- Функции ---
def install_custom_node(url):
    %cd /content/ComfyUI/custom_nodes
    !git clone {url}

def downloadModel(url, subdir, filename=None):
    target_dir = f"{MODELS_DIR}/{subdir}"
    os.makedirs(target_dir, exist_ok=True)

    if 'huggingface.co' in url:
        if filename is None:
            filename = url.split('/')[-1].removesuffix('?download=true')
        full_url = url
    else:  # civitai
        full_url = f"{url}?token={CIVITAI_API_TOKEN}"
        if filename is None:
            filename = None

    # пропускаем, если файл уже полностью скачан
    if filename:
        path = f"{target_dir}/{filename}"
        if os.path.exists(path) and not os.path.exists(path + ".aria2"):
            print(f"✅ Уже есть: {filename}")
            return
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d "{target_dir}" -o "{filename}" "{full_url}"
    else:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d "{target_dir}" "{full_url}"

# --- Кастомные ноды (по желанию) ---
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Impact-Pack')

# --- Модели Wan 2.2 ---
HF = "https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files"

downloadModel(f"{HF}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors?download=true", "diffusion_models")
downloadModel(f"{HF}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors?download=true", "diffusion_models")
downloadModel(f"{HF}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors?download=true", "text_encoders")
downloadModel(f"{HF}/vae/wan2.2_vae.safetensors?download=true", "vae")

print("🎉 Готово")

/content/ComfyUI
Selecting previously unselected package libcares2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symb

## OTHERS

In [1]:
# @title
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

--2026-11-13 14:52:19--  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.40, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt [following]
--2026-11-13 14:52:19--  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66d19580e2632490a6bc5829/773d9e5ddbf6a84612066b3ea00a0bcfe4ba870a9befc88816f71b5927e76e38?X-Xet-Cas-Uid=public&user_id=public&xip=MjT2lSg3P7Q&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27v1-5-pruned-emaonly.ckpt%3B+filename%3D%22v1-5-pruned-emaonly

## LIST MODELS

In [4]:
!ls -al ./models/diffusion_models/

total 27919440
drwxr-xr-x  2 root root        4096 Nov 13 17:35 .
drwxr-xr-x 10 root root        4096 Nov 13 17:33 ..
-rw-r--r--  1 root root 14294742832 Nov 13 17:34 wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors
-rw-r--r--  1 root root 14294742832 Nov 13 17:35 wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [7]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-11-13 17:38:48--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.247.166.64
Connecting to github.com (github.com)|20.247.166.64|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64.deb [following]
--2026-11-13 17:38:48--  https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/04372d98-d20f-4427-9a16-58016e29739c?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-13T18%3A26%3A45Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b

In [8]:
!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'closest_hit_on_mesh', 'closest_point_on_mesh', 'connected_components', 'convrot_w4a4_linear', 'delaunay3d', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'dequantize_w4a8_int8_weig

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [9]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-11-13 17:41:00--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 57.155.223.240
Connecting to github.com (github.com)|57.155.223.240|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64.deb [following]
--2026-11-13 17:41:01--  https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/04372d98-d20f-4427-9a16-58016e29739c?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-13T18%3A26%3A13Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b89

## CF Tunnel

In [10]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'na3d', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'rms_adaln', 'rms_rope', 'rms_rope1', 'rms_rope1_', 'rms_rope_', 'rms_rope_split_half', 'rms_rope_split_half1', 'rms_rope_split_half1_', 'rms_rope_split_half_

## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server